# YouTube Video Q&A — RAG Chatbot

A Retrieval-Augmented Generation pipeline that answers questions about a YouTube video, grounded strictly in its transcript.

## 1. Setup

Install dependencies and load API key from a `.env` file (never hardcode keys).

In [ ]:
!pip install -q langchain youtube-transcript-api langchain-community langchain-groq faiss-cpu python-dotenv langchain-huggingface sentence-transformers langchain-text-splitters


In [ ]:
from dotenv import load_dotenv
load_dotenv()  # expects GROQ_API_KEY to be set in a local .env file (see README)


In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, NoTranscriptFound, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser


## 2. Indexing — Document Ingestion

Fetch the transcript for a given YouTube video (auto-translates to English if needed).

In [ ]:
video_id = "p4qPp_3_f2Y"  # replace with the target video's ID (the part after v= in the URL)

try:
    yt = YouTubeTranscriptApi()
    transcript_list = yt.list(video_id)

    try:
        _transcript_obj = transcript_list.find_manually_created_transcript(
            [t.language_code for t in transcript_list]
        )
    except NoTranscriptFound:
        _transcript_obj = next(iter(transcript_list))

    _original_lang = _transcript_obj.language_code

    if _original_lang != 'en':
        _transcript_obj = _transcript_obj.translate('en')

    fetched = _transcript_obj.fetch()
    transcript = ' '.join(chunk.text for chunk in fetched)

    print(f"Original language: {_original_lang} → translated to English")
    print("Transcript length:", len(transcript))

except (TranscriptsDisabled, NoTranscriptFound) as e:
    print(f"Could not get transcript: {e}")


## 3. Indexing — Text Splitting

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])
print("Number of chunks:", len(chunks))


## 4. Indexing — Embedding & Vector Store

Embeds each chunk with a sentence-transformer model and stores the vectors in FAISS for similarity search.

> Set `device` to `'cpu'` if you don't have a CUDA GPU available.

In [ ]:
embedding = HuggingFaceEmbeddings(
    model="BAAI/bge-base-en-v1.5",
    encode_kwargs={'normalize_embeddings': True},
    model_kwargs={'device': 'cuda'}  # change to 'cpu' if no GPU is available
)

vector_store = FAISS.from_documents(chunks, embedding)


## 5. Retriever

In [ ]:
retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k': 5})


## 6. Augmentation — Prompt Template

In [ ]:
llm = ChatGroq(
    model='openai/gpt-oss-120b',
    temperature=0.3
)

prompt = PromptTemplate(
    template='''
    You are a helpful assistant.
    Answer ONLY from the provided transcript context.
    If the context is insufficient, just say "Nothing related to your question --> {question} is discussed in the video."

    {context}
    Question: {question}
    ''',
    input_variables=['question', 'context']
)


### Quick single-question test (manual retrieval)

In [ ]:
question = "What is nuclear fusion?"
retrieved_docs = retriever.invoke(question)
context_text = '\n\n'.join(doc.page_content for doc in retrieved_docs)
final_prompt = prompt.invoke({"question": question, "context": context_text})


## 7. Generation

In [ ]:
answer = llm.invoke(final_prompt)
print(answer.content)


## 8. Full RAG Chain (LCEL)

Wires retrieval, prompting, generation, and output parsing into a single composable chain.

In [ ]:
def format_docs(retrieved_docs):
    return "\n\n".join(doc.page_content for doc in retrieved_docs)


parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

parser = StrOutputParser()

main_chain = parallel_chain | prompt | llm | parser


### Ask a question

In [ ]:
main_chain.invoke("Summarize the main topic of this video.")
